# token_manager

`SpotifyAuth` — unified class for the OAuth2 Authorization Code flow and token
lifecycle (load, refresh, auto-expire).

**Used by other notebooks via `%run`.**  
Call `get_access_token()` to obtain a valid Bearer token without worrying about expiry.

Dependencies:
- `%run ../config/settings`

In [ ]:
# %run ../config/settings

In [ ]:
import base64
import json
import time
import urllib.error
import urllib.parse
import urllib.request


class SpotifyAuth:
    """
    Handles Spotify OAuth2 Authorization Code flow and token lifecycle.

    Two usage modes:
      1. First-time setup (interactive):
           auth = SpotifyAuth(client_id, client_secret, redirect_uri)
           print(auth.get_authorization_url())   # open in browser
           auth.exchange_code("<code from callback>")
           print(auth.refresh_token)             # persist to Databricks secrets

      2. Runtime (automated pipelines):
           auth = SpotifyAuth(client_id, client_secret, redirect_uri)
           auth.load_refresh_token("<stored refresh_token>")
           token = auth.get_access_token()       # auto-refreshes when near expiry
    """

    def __init__(
        self,
        client_id: str,
        client_secret: str,
        redirect_uri: str,
        scopes: list | None = None,
    ):
        self.client_id     = client_id
        self.client_secret = client_secret
        self.redirect_uri  = redirect_uri
        self.scopes        = scopes or DEFAULT_SCOPES

        self._access_token:  str | None = None
        self._refresh_token: str | None = None
        self._expires_at:    float      = 0.0

    # ── Properties ────────────────────────────────────────────────────────────

    @property
    def refresh_token(self) -> str | None:
        return self._refresh_token

    # ── OAuth2 Authorization Code flow ────────────────────────────────────────

    def get_authorization_url(self) -> str:
        """Return the Spotify URL the user must visit to grant access."""
        params = {
            "client_id":     self.client_id,
            "response_type": "code",
            "redirect_uri":  self.redirect_uri,
            "scope":         " ".join(self.scopes),
        }
        return f"{SPOTIFY_AUTH_URL}?{urllib.parse.urlencode(params)}"

    def exchange_code(self, authorization_code: str) -> None:
        """Exchange a one-time authorization code for refresh + access tokens."""
        data = self._post_token({
            "grant_type":   "authorization_code",
            "code":          authorization_code,
            "redirect_uri":  self.redirect_uri,
        })
        self._store(data)

    # ── Token lifecycle ───────────────────────────────────────────────────────

    def load_refresh_token(self, refresh_token: str) -> None:
        """Seed the instance with a persisted refresh_token (skip OAuth flow)."""
        self._refresh_token = refresh_token

    def get_access_token(self) -> str:
        """Return a valid Bearer token, auto-refreshing if within 60 s of expiry."""
        if self._needs_refresh():
            self._do_refresh()
        return self._access_token  # type: ignore[return-value]

    def invalidate(self) -> None:
        """Force the next get_access_token() call to refresh (use after 401)."""
        self._access_token = None
        self._expires_at   = 0.0

    # ── Internals ─────────────────────────────────────────────────────────────

    def _needs_refresh(self) -> bool:
        return self._access_token is None or time.time() >= self._expires_at - 60

    def _do_refresh(self) -> None:
        if not self._refresh_token:
            raise RuntimeError(
                "No refresh_token available. "
                "Run the oauth_flow notebook first and store the token in Databricks secrets."
            )
        data = self._post_token({
            "grant_type":    "refresh_token",
            "refresh_token":  self._refresh_token,
        })
        self._store(data)

    def _store(self, data: dict) -> None:
        self._access_token = data["access_token"]
        self._expires_at   = time.time() + int(data["expires_in"])
        # Spotify only returns a new refresh_token on code exchange, not on refresh
        if "refresh_token" in data:
            self._refresh_token = data["refresh_token"]

    def _post_token(self, body: dict) -> dict:
        creds = base64.b64encode(
            f"{self.client_id}:{self.client_secret}".encode()
        ).decode()
        req = urllib.request.Request(
            SPOTIFY_TOKEN_URL,
            data=urllib.parse.urlencode(body).encode(),
            headers={
                "Authorization": f"Basic {creds}",
                "Content-Type":  "application/x-www-form-urlencoded",
            },
        )
        try:
            with urllib.request.urlopen(req) as resp:
                return json.loads(resp.read())
        except urllib.error.HTTPError as e:
            raise RuntimeError(
                f"Token request failed {e.code}: {e.read().decode()}"
            ) from e

In [ ]:
# ── Module-level singleton ────────────────────────────────────────────────────
# Shared across all notebooks in the session that %run this file.

_auth_singleton: SpotifyAuth | None = None


def _init_auth(secret_scope: str = SECRET_SCOPE) -> SpotifyAuth:
    client_id     = dbutils.secrets.get(secret_scope, "spotify_client_id")
    client_secret = dbutils.secrets.get(secret_scope, "spotify_client_secret")
    redirect_uri  = dbutils.secrets.get(secret_scope, "spotify_redirect_uri")
    refresh_token = dbutils.secrets.get(secret_scope, "spotify_refresh_token")

    auth = SpotifyAuth(client_id, client_secret, redirect_uri)
    auth.load_refresh_token(refresh_token)
    return auth


def get_access_token(secret_scope: str = SECRET_SCOPE) -> str:
    """Public API: return a valid Spotify access token, refreshing when needed."""
    global _auth_singleton
    if _auth_singleton is None:
        _auth_singleton = _init_auth(secret_scope)
    return _auth_singleton.get_access_token()